In [ ]:
import read_eps
import  matplotlib.pyplot as plt
import numpy as np
import lab
import potcorr
import const

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


cell=lab.mos2_12to24_irrbz
ttkw=potcorr.PotCorr(cell)
ttkw.fft_init()
ttkw.get_vcoul()

#ttkw.read_chi(cell['folder']+'../chi_24x24_fermi_-0.15/chimat.h5',cell['folder']+'../chi_24x24_fermi_-0.15/chimat_sq.h5' )
#ttkw.read_chi(cell['folder']+'../chi_24x24/chimat.h5',cell['folder']+'../chi_24x24/chi0mat.h5' )
eps1 = '/anvil/scratch/x-rg47749/data/mos2/2DEG/conc_1e13_q2DEG/epsmat.h5'
eps0 = '/anvil/scratch/x-rg47749/data/mos2/2DEG/conc_1e13_q2DEG/eps0mat.h5'
ttkw.read_epsinv(eps1=eps1, eps0=eps0)
#ttkw.read_epsinv(cell['folder']+'../eps_inv_24x24_fermi_-0.25_2d/epsmat_inv.h5',cell['folder']+'../eps_inv_24x24_fermi_-0.25_2d/epsmat_sq.h5' )
#ttkw.epsmat_init()
k_symmetry_map =  ttkw.get_k_symmetry_map()
#epsym_dict = ttkw.get_epsym_dict(k_symmetry_map, ecut=12)
epsym_dict = ttkw.read_epsym_dict('/anvil/scratch/x-rg47749/data/mos2/symm/epsym_dict.pkl')  #
#print(mos2.q_ind_tt)
#ttkw.get_epsinv_mat_irrbz(G_ind_cut = 1000)
#ttkw.epsmat_inv(G_ind_cut = 600)


In [ ]:
epsmat_eps2rho_dict_file = '/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_eps2rho_dict.pkl'
epsmat_eps2eps_irrbz_dict_file ='/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_eps2eps_irrbz_dict.pkl'
epsmat_dict_file = '/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_dict.pkl'
epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict= ttkw.read_epsmat_dict(epsmat_eps2rho_dict_file, epsmat_eps2eps_irrbz_dict_file, epsmat_dict_file)

In [ ]:
wcoul_mat_dict_file = '/anvil/scratch/x-rg47749/data/mos2/symm/wcoul_mat_dict.pkl'
wcoul_mat_dict= ttkw.read_wcoul_mat_dict(wcoul_mat_dict_file)

In [ ]:
rho_ext_r = np.load('/anvil/scratch/x-rg47749/data/mos2/12x12/rho_bare_12-24.npy')
rho_ext_k = np.fft.fftn(rho_ext_r)

In [ ]:
phi_G_dict = ttkw.gen_phi_G_dict(rho_ext_k, k_symmetry_map, epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, wcoul_mat_dict)

In [ ]:
ttkw.vcoul2d0modify()

In [ ]:
q_vec= (0.0,0.0,0.0)
epsmat_dict[q_vec]=(ttkw.Eps0.mat[0,0,0,:,:,0]+ttkw.Eps0.mat[0,0,0,:,:,1]*1j).T
epsmat_eps2rho_dict[q_vec]=ttkw.Eps0.gind_eps2rho[0,:ttkw.Eps0.nmtx[0]]-1
epsmat_eps2eps_irrbz_dict[q_vec]=np.arange(ttkw.Eps0.nmtx[0])

In [ ]:
wcoul_mat_dict = ttkw.gen_wcoul_mat_dict(k_symmetry_map,epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict )

In [ ]:
wcoul_mat=np.zeros_like(epsmat_dict[q_vec])
for eps_id, rho_id in enumerate(epsmat_eps2rho_dict[q_vec]):  
    g_vec =  ttkw.Eps0.G_ind2vec[rho_id]
    print(eps_id, rho_id, g_vec)
    x_ind = np.where(np.round(ttkw.fft_kx_tt, 8) == q_vec[0]+g_vec[0])[0][0]
    y_ind = np.where(np.round(ttkw.fft_ky_tt, 8) == q_vec[1]+g_vec[1])[0][0]
    z_ind = np.where(np.round(ttkw.fft_kz_tt, 8) == q_vec[2]+g_vec[2])[0][0]
    v_coul_tmp = ttkw.v_coul2d[x_ind, y_ind, z_ind]
    print(v_coul_tmp)
    wcoul_mat[:, eps_id]=epsmat_dict[q_vec][:,eps_id]*v_coul_tmp

In [ ]:
q_vec = (0.0,0.0,0.0)
phi_G = np.zeros_like(wcoul_mat_dict[q_vec][:,0])
for eps_id, rho_id in enumerate(epsmat_eps2rho_dict[q_vec]):  
    g_vec =  ttkw.Eps0.G_ind2vec[rho_id]
    print(eps_id, rho_id, g_vec)
    x_ind = np.where(np.round(ttkw.fft_kx_tt, 8) == q_vec[0]+g_vec[0])[0][0]
    y_ind = np.where(np.round(ttkw.fft_ky_tt, 8) == q_vec[1]+g_vec[1])[0][0]
    z_ind = np.where(np.round(ttkw.fft_kz_tt, 8) == q_vec[2]+g_vec[2])[0][0]
    rho_tmp = rho_ext_k[x_ind, y_ind, z_ind]
    phi_G += wcoul_mat_dict[q_vec][:,eps_id]*rho_tmp

In [ ]:
q_vec = (0.0,0.0,0.0)
phi_G_dict[q_vec]
phi_k = np.zeros((ttkw.fft_nx,ttkw.fft_ny,ttkw.fft_nz))
for eps_id, rho_id in enumerate(epsmat_eps2rho_dict[q_vec]):  
    g_vec =  ttkw.Eps0.G_ind2vec[rho_id]
    print(eps_id, rho_id, g_vec)
    x_ind = np.where(np.round(ttkw.fft_kx_tt, 8) == q_vec[0]+g_vec[0])[0][0]
    y_ind = np.where(np.round(ttkw.fft_ky_tt, 8) == q_vec[1]+g_vec[1])[0][0]
    z_ind = np.where(np.round(ttkw.fft_kz_tt, 8) == q_vec[2]+g_vec[2])[0][0]
    phi_k[x_ind, y_ind, z_ind]=phi_G_dict[q_vec][eps_id]

In [ ]:
phi_k = np.zeros((ttkw.fft_nx,ttkw.fft_ny,ttkw.fft_nz), dtype=complex)
for q_vec in phi_G_dict.keys():
    print(f'mapping {q_vec} to phi_k')
    for eps_id, rho_id in enumerate(epsmat_eps2rho_dict[q_vec]):  
        g_vec =  ttkw.Eps1.G_ind2vec[rho_id]
        #print(eps_id, rho_id, g_vec)
        x_ind = np.where(np.round(ttkw.fft_kx_tt, 8) == np.round((q_vec[0]+g_vec[0]),8))[0][0]
        y_ind = np.where(np.round(ttkw.fft_ky_tt, 8) == np.round((q_vec[1]+g_vec[1]),8))[0][0]
        z_ind = np.where(np.round(ttkw.fft_kz_tt, 8) == np.round((q_vec[2]+g_vec[2]),8))[0][0]
        phi_k[x_ind, y_ind, z_ind]=phi_G_dict[q_vec][eps_id]

In [ ]:
phi_r = np.fft.ifftn(phi_k)

In [ ]:
#Eps1 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/chi_24x24/chimat.h5')
#Eps0 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/chi_24x24_fermi_-0.15/chimat_sq.h5')
#Eps1 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/eps_inv_24x24_fermi_-0.15_2d/epsmat_inv.h5')
#Eps0 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/eps_inv_24x24_fermi_-0.15_2d/epsmat_sq.h5')
Eps1 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/eps_inv_24x24/epsmat.h5')
Eps0 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/eps_inv_24x24/eps0mat.h5')

#Eps1 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/chi_24x24_int/chimat.h5')
#Eps0 =  read_eps.Epsmat('/anvil/scratch/x-rg47749/data/mos2/chi_24x24_int/chi0mat.h5')

In [ ]:
def get_qxy(qpts):
    qxy = np.sqrt(qpts[:,0]**2 + (np.sqrt(3)/3*qpts[:,0]+np.sqrt(3)*2/3*qpts[:,1])**2)
    return qxy

In [ ]:
fine_data = rho_ext_r
f_transform = np.fft.fftn(fine_data)

# 获取原始数据的维度
nx, ny, nz = f_transform.shape

# 创建一个用0填充的更大的傅里叶空间数组
new_f_transform = np.zeros((180,180,60), dtype=complex)
nx, ny, nz = new_f_transform.shape
# 将原始的傅里叶变换数据复制到新的傅里叶空间中心
new_f_transform[:nx//2, :ny//2, :nz//2] = f_transform[:nx//2, :ny//2, :nz//2]
new_f_transform[-nx//2:, :ny//2, :nz//2] = f_transform[-nx//2:, :ny//2, :nz//2]
new_f_transform[:nx//2, -ny//2:, :nz//2] = f_transform[:nx//2, -ny//2:, :nz//2]
new_f_transform[-nx//2:, -ny//2:, :nz//2] = f_transform[-nx//2:, -ny//2:, :nz//2]
new_f_transform[:nx//2, :ny//2, -nz//2:] = f_transform[:nx//2, :ny//2, -nz//2:]
new_f_transform[-nx//2:, :ny//2, -nz//2:] = f_transform[-nx//2:, :ny//2, -nz//2:]
new_f_transform[:nx//2, -ny//2:, -nz//2:] = f_transform[:nx//2, -ny//2:, -nz//2:]
new_f_transform[-nx//2:, -ny//2:, -nz//2:] = f_transform[-nx//2:, -ny//2:, -nz//2:]

# 执行逆傅里叶变换并缩放（因为数组大小已经改变）
fine_data2 = np.fft.ifftn(new_f_transform).real *( 180/720*180/720*60/225)

In [ ]:
print(np.sum(fine_data)/720/720/225*ttkw.omega)
print(np.sum(fine_data2)/180/180/60*ttkw.omega)
rho_ext_k = np.fft.fftn(fine_data2)

In [ ]:
pot_tot = ttkw.rho_ext2pot_tot_irrbz(epsym_dict,rho_ext_k, G_ind_cut=400)

In [ ]:
#from sklearn.preprocessing import PolynomialFeatures
#from sklearn.linear_model import LinearRegression
#from sklearn.pipeline import make_pipeline

for gg in range(0,1):
    g0_ = [-1,-1,1]
    g1_ = [-1,-1,1]
    #q0_ = Eps0.qpts[:]
    q1_ = np.array(list(sorted(epsym_dict.keys())))
    #q0_abs_ = np.sqrt((q0_[:,0])**2+ (np.sqrt(3)/3*q0_[:,0]+2*np.sqrt(3)/3*q0_[:,1])**2)
    q1_abs_ = np.sqrt((q1_[:,0])**2+ (np.sqrt(3)/3*q1_[:,0]+2*np.sqrt(3)/3*q1_[:,1])**2)
    epsinv_Re_ = []
    epsinv_Im_ = []
    eps_Re_ = []
    eps_Im_ = []
    wcoul_Re_ = []
    wcoul_Im_ = []
    chi_Re_ = [] 
    chi_Im_ = []
    q_abs_ = []
    for i in range(576):
        q_vec_ = q1_[i]
        if i==0:
            qind_ = i
            g_vec0_ = tuple(g0_)
            g_vec1_ = tuple(g1_)
            gind_rho0_ = ttkw.Eps0.G_vec2ind[g_vec0_]
            gind_rho1_ = ttkw.Eps0.G_vec2ind[g_vec1_]
           
            gind_eps0_ = ttkw.Eps0.gind_rho2eps[qind_, gind_rho0_]
            gind_eps1_ = ttkw.Eps0.gind_rho2eps[qind_, gind_rho1_]
            
            mat_Re_ = ttkw.epsinv_mat[qind_,gind_eps0_-1,gind_eps1_-1].real
            mat_Im_ = ttkw.epsinv_mat[qind_,gind_eps0_-1,gind_eps1_-1].imag
            epsinv_Re_.append(mat_Re_)
            epsinv_Im_.append(mat_Im_)

            mat_Re_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].real
            mat_Im_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].imag
            wcoul_Re_.append(mat_Re_)
            wcoul_Im_.append(mat_Im_)

        else:
            qind_ = i
            g_vec0_ = tuple(g0_)
            g_vec1_ = tuple(g1_)
            gind_rho0_ = ttkw.Eps1.G_vec2ind[g_vec0_]
            gind_rho1_ = ttkw.Eps1.G_vec2ind[g_vec1_]
            #gind_eps0_ = ttkw.Eps1.gind_rho2eps[qind_, gind_rho0_]
            #gind_eps1_ = ttkw.Eps1.gind_rho2eps[qind_, gind_rho1_]
            gind_eps0_ = epsym_dict[potcorr.round_tuple(q_vec_, 8)][gind_rho0_]
            gind_eps1_ = epsym_dict[potcorr.round_tuple(q_vec_, 8)][gind_rho1_]

            mat_Re_ = ttkw.epsinv_mat[qind_,gind_eps0_,gind_eps1_].real
            mat_Im_ = ttkw.epsinv_mat[qind_,gind_eps0_,gind_eps1_].imag
            epsinv_Re_.append(mat_Re_)
            epsinv_Im_.append(mat_Im_)

            mat_Re_ = ttkw.wcoul[qind_,gind_eps0_,gind_eps1_].real
            mat_Im_ = ttkw.wcoul[qind_,gind_eps0_,gind_eps1_].imag
            wcoul_Re_.append(mat_Re_)
            wcoul_Im_.append(mat_Im_)
        
     #   mat_Re_ = ttkw.chi_mat[qind_,gind_eps0_-1,gind_eps1_-1].real
     #   mat_Im_ = ttkw.chi_mat[qind_,gind_eps0_-1,gind_eps1_-1].imag
     #   chi_Re_.append(mat_Re_)
     #   chi_Im_.append(mat_Im_)
        
     #   mat_Re_ = ttkw.eps_mat[qind_,gind_eps0_-1,gind_eps1_-1].real
     #   mat_Im_ = ttkw.eps_mat[qind_,gind_eps0_-1,gind_eps1_-1].imag
     #   eps_Re_.append(mat_Re_)
     #   eps_Im_.append(mat_Im_)
    q_abs_ = np.hstack((q_abs_, q1_abs_ ))
    
#fig, ax = plt.subplots(figsize = (12,6), dpi = 100)
#ax.scatter(q_abs_, np.array(eps_Re_))
#ax.scatter(q_abs_, eps_Im_)
#plt.show()

fig, ax = plt.subplots(figsize = (12,6), dpi = 100)
ax.scatter(q_abs_, epsinv_Re_)
ax.scatter(q_abs_, epsinv_Im_)
plt.title('Inverse epsilon')
plt.show()
fig, ax = plt.subplots(figsize = (12,6), dpi = 100)
ax.scatter(q_abs_, wcoul_Re_)
ax.scatter(q_abs_, wcoul_Im_)
plt.title('Screened Coulomb interaction w')
plt.show()

In [ ]:
eps_re = np.array(eps_Re_)

In [ ]:
len( np.array(eps_Re_))

In [ ]:
ttkw.v_coul[1,0,0]

In [ ]:
ttkw.lattpara_unit

In [ ]:
0.01202813

In [ ]:
for gg in range(0,1):
    g0_ = [0,0,0]
    g1_ = [0,0,0]
    q0_ = Eps0.qpts[:]
    q1_ = Eps1.qpts[:]
    q0_abs_ = np.sqrt((q0_[:,0])**2+ (np.sqrt(3)/3*q0_[:,0]+2*np.sqrt(3)/3*q0_[:,1])**2)
    print(q0_abs_)
    q1_abs_ = np.sqrt((q1_[:,0])**2+ (np.sqrt(3)/3*q1_[:,0]+2*np.sqrt(3)/3*q1_[:,1])**2)
    eps_Re_ = []
    eps_Im_ = []
    wcoul_Re_ = []
    wcoul_Im_ = []
    q_abs_ = []
    for i in range(10):
        qind_ = i
        g_vec0_ = tuple(g0_)
        g_vec1_ = tuple(g1_)
        gind_rho0_ = Eps0.G_vec2ind[g_vec0_]
        gind_rho1_ = Eps0.G_vec2ind[g_vec1_]
        gind_eps0_ = Eps0.gind_rho2eps[qind_, gind_rho0_]
        gind_eps1_ = Eps0.gind_rho2eps[qind_, gind_rho1_]
        
        mat_Re_ = Eps0.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,0]
        mat_Im_ = Eps0.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,1]
        eps_Re_.append(mat_Re_)
        eps_Im_.append(mat_Im_)
        q_abs_of_ = np.sqrt((q0_[i,0])**2+ (np.sqrt(3)/3*q0_[i,0]+2*np.sqrt(3)/3*q0_[i,1])**2)
        q_abs_ = np.hstack((q_abs_, q_abs_of_))
        
    for i in range(1,576):
        qind_ = i
        g_vec0_ = tuple(g0_)
        g_vec1_ = tuple(g1_)
        gind_rho0_ = Eps1.G_vec2ind[g_vec0_]
        gind_rho1_ = Eps1.G_vec2ind[g_vec1_]
        gind_eps0_ = Eps1.gind_rho2eps[qind_, gind_rho0_]
        gind_eps1_ = Eps1.gind_rho2eps[qind_, gind_rho1_]
        
        mat_Re_ = Eps1.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,0]
        mat_Im_ = Eps1.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,1]
        eps_Re_.append(mat_Re_)
        eps_Im_.append(mat_Im_)
        
        q_abs_of_ = np.sqrt((q1_[i,0])**2+ (np.sqrt(3)/3*q1_[i,0]+2*np.sqrt(3)/3*q1_[i,1])**2)
        q_abs_ = np.hstack((q_abs_, q_abs_of_))
        
        #mat_Re_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].real
        #mat_Im_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].imag
        #wcoul_Re_.append(mat_Re_)
        #wcoul_Im_.append(mat_Im_)
    #q_abs_ = np.hstack((q0_abs_, q1_abs_ ))
    
    fig, ax = plt.subplots(figsize = (12,6), dpi = 100)
    ax.scatter(q_abs_, np.array(eps_Re_))
    ax.scatter(q_abs_, eps_Im_)
    # ax.scatter(q_abs_, eps_re)
    ax.scatter(q_abs_[:10], eps_Re_[:10])
    #plt.xlim(-0.01,0.4)
    plt.show()


In [ ]:
for gg in range(0,1):
    g0_ = [0,0,0]
    g1_ = [0,0,0]
    q0_ = Eps0.qpts[:]
    q1_ = Eps1.qpts[:]
    q0_abs_ = np.sqrt((q0_[:,0])**2+ (np.sqrt(3)/3*q0_[:,0]+2*np.sqrt(3)/3*q0_[:,1])**2)
    q1_abs_ = np.sqrt((q1_[:,0])**2+ (np.sqrt(3)/3*q1_[:,0]+2*np.sqrt(3)/3*q1_[:,1])**2)
    eps_Re_ = []
    eps_Im_ = []
    wcoul_Re_ = []
    wcoul_Im_ = []
    q_abs_ = []
    for i in range(6):
        qind_ = i
        g_vec0_ = tuple(g0_)
        g_vec1_ = tuple(g1_)
        gind_rho0_ = Eps0.G_vec2ind[g_vec0_]
        gind_rho1_ = Eps0.G_vec2ind[g_vec1_]
        gind_eps0_ = Eps0.gind_rho2eps[qind_, gind_rho0_]
        gind_eps1_ = Eps0.gind_rho2eps[qind_, gind_rho1_]
        
        mat_Re_ = Eps0.mat[qind_,0,0,gind_eps0_-1,gind_eps1_-1,0]
        mat_Im_ = Eps0.mat[qind_,0,0,gind_eps0_-1,gind_eps1_-1,1]
        eps_Re_.append(mat_Re_)
        eps_Im_.append(mat_Im_)
    q_abs_ = np.hstack((q_abs_,q0_abs_ ))
    fig, ax = plt.subplots(figsize = (12,6), dpi = 100)


    ax.scatter(q_abs_, eps_Re_)
    ax.scatter(q_abs_, eps_Im_)

    plt.show()


In [ ]:
G_ind_list_tmp = ttkw.g_epsind_tt[ttkw.q_ind_tt == 1]

In [ ]:
G_ind_list_tmp

In [ ]:
v_coul_list_tmp = ttkw.v_coul[ttkw.q_ind_tt == 1]

In [ ]:
v_coul_tmp = v_coul_list_tmp[G_ind_list_tmp == 129]

In [ ]:
pot_tot_r = np.fft.ifftn(pot_tot)

In [ ]:
pot_tot_r = np.load('/anvil/scratch/x-rg47749/data/mos2/12x12/pot_tot_by_wcoul_12-24_2d.npy')
pot2_tot_r = np.load('/anvil/scratch/x-rg47749/data/mos2/12x12/pot_tot_by_wcoul_12-24_2d_fermi_-0.25.npy')

In [ ]:

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_tot_r.real[:,:,ttkw.fft_nz//2+10], 
                #gridsize=100,
                vmax=0.055,
                vmin=-0.005,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_tot_r.real[:,:,ttkw.fft_nz//2-10], 
                #gridsize=100,
                vmax=0.055,
                vmin=-0.005,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot2_tot_r.real[:,:,ttkw.fft_nz//2+10], 
                #gridsize=100,
                vmax=0.055,
                vmin=-0.005,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot2_tot_r.real[:,:,ttkw.fft_nz//2-10], 
                #gridsize=100,
                vmax=0.055,
                vmin=-0.005,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
#plt.plot(range(720), np.sum(pot_tot_r.real[:,:,225//2-5:225//2+5], axis=(1,2)))
#plt.ylim(-0.001,0.001)
plt.plot(range(720), phi_r.real[:,720//2,225//2])
#plt.plot(range(720), pot2_tot_r.real[:,720//2,225//2])

In [ ]:
plt.plot(range(225), phi_r.real[ttkw.fft_nx//2,ttkw.fft_ny//2,:])

In [ ]:
plt.plot(range(225), np.sum(phi_r[:,:,:],axis=(0,1)))

In [ ]:
np.save(cell['folder']+'pot_fermi_0.02/phi_r_aligned.npy',pot_tot_aligned.real)

In [ ]:
pot_tot_r.real[360,360,100]

In [ ]:
v_coul_r = np.fft.ifftn(ttkw.v_coul).real
v_coul2d_r = np.fft.ifftn(ttkw.v_coul2d).real

In [ ]:
plt.scatter(range(len(ttkw.v_coul2d[:,0,0])),ttkw.v_coul2d[:,0,0])
plt.scatter(range(len(ttkw.v_coul[:,0,0])),ttkw.v_coul[:,0,0])
plt.xlim(0,20)

In [ ]:
plt.plot(range(len(ttkw.v_coul2d[0,0,:])),np.sum(v_coul_r[:,:,:],axis=(0,1)))
plt.plot(range(len(ttkw.v_coul2d[0,0,:])),np.sum(v_coul2d_r[:,:,:],axis=(0,1)))
#plt.scatter(range(len(ttkw.v_coul[0,0,:])),ttkw.v_coul[:,0,0])

In [ ]:
plt.plot(range(len(ttkw.v_coul2d[0,0,:])),v_coul_r[ttkw.fft_nx//2+100,ttkw.fft_ny//2+100,:])
#plt.plot(range(len(ttkw.v_coul2d[0,0,:])),v_coul2d_r[ttkw.fft_nx//2+100,ttkw.fft_ny//2+100,:])

In [ ]:
plt.plot(range(50),np.sum(rho_ext_r[:,:,-50:],axis=(0,1)))

In [ ]:
cmap = plt.cm.get_cmap('coolwarm', 225//2)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(225//2):
    ax.plot(range(720), phi_r.real[:, 720//2, 225//2+i], color=cmap(np.abs(i)))
#plt.ylim(-0.002, 0.003)
plt.show()

In [ ]:
plt.plot(range(225),np.average(phi_r[:10,:10,:], axis = (0,1)))
plt.plot(range(225),np.average(phi_r[:50,:50,:], axis = (0,1)))
plt.plot(range(225),np.average(phi_r[:100,:100,:], axis = (0,1)))


In [ ]:
pot_align_z = np.average(phi_r[:50,:50,:], axis = (0,1))

In [ ]:
pot_tot_aligned = np.zeros_like(phi_r)
for i in range(ttkw.fft_nz):
    pot_tot_aligned[:,:,i] = phi_r[:,:,i] - pot_align_z[i]

In [ ]:
cmap = plt.cm.get_cmap('coolwarm', 225//2)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(225//2):
    ax.plot(range(720), pot_tot_aligned[:, 720//2, 225//2+i], color=cmap(np.abs(i)))
#plt.ylim(-0.002, 0.003)
plt.show()

In [ ]:
plt.plot(range(225), np.sum(pot_tot_aligned[:,:,:],axis=(0,1)))
plt.plot(range(225), np.sum(phi_r[:,:,:],axis=(0,1)))

In [ ]:
plt.plot(range(720), np.sum(pot_tot_aligned.real[:,:,225//2-250:225//2+250], axis=(1,2)))

In [ ]:
pot_tot_r = np.load('/anvil/scratch/x-rg47749/data/mos2/12x12/pot_tot_by_wcoul_12-24_2d_aligned.npy')

In [ ]:
import utli
#nx, ny, nz = ttkw.fft_nx, ttkw.fft_ny, ttkw.fft_nz
nx, ny, nz = pot_tot_r.shape
potential_tot4 = pot2_tot_r[:,:,int(0.48*nz):int(0.52*nz)]
potential_tot5 = pot_tot_r[:,:,int(0.48*nz):int(0.52*nz)]#/1.01213
#potential_tot4 = pot_model4[:,:,int(0.45*nz):int(0.55*nz)]
A = float(ttkw.lattpara[0])
C = 0.04*float(ttkw.lattpara[2])

print(A)
print(nz)


dist_arr_tot = utli.distance_array(potential_tot5.shape[0], potential_tot5.shape[1], potential_tot5.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

#dist_arr_tot = utli.distance_array(potential_tot.shape[0], potential_tot.shape[1], potential_tot.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

fig, ax = plt.subplots(figsize=(12,6),dpi=100)

ax.scatter(dist_arr_tot.flatten(), potential_tot5.flatten(),alpha=0.01, label='fermi -0.15')
ax.scatter(dist_arr_tot.flatten(), potential_tot4.flatten(),alpha=0.01, label='fermi -0.25')
#ax.scatter(dist_arr_tot.flatten(), potential_tot4.flatten(), label='fermi -0.15')
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten()*1.05, label='model with interp')
#plt.scatter(dist_arr_dv.flatten(), potential_dv.flatten(), label='dft')
#plt.scatter(dist_arr_tot[:,:,11].flatten(), potential_tot[:,:,11].flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,6].flatten(), potential_tot[:,:,6].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
#plt.scatter(dist_arr_tot[:,:,0].flatten(), potential_tot[:,:,0].flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect (Bohr)')
plt.ylabel('Potential (Ry)')
plt.legend()
plt.show()

In [ ]:
import pickle

with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsym_dict.pkl', 'wb') as f:
    pickle.dump(epsym_dict, f)

In [ ]:
import pickle

with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_eps2rho_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_eps2rho_dict, f)
with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_eps2eps_irrbz_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_eps2eps_irrbz_dict, f)
with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_dict, f)

In [ ]:
import pickle

with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_eps2eps_irrbz_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_eps2eps_irrbz_dict, f)

In [ ]:
import pickle

with open('/anvil/scratch/x-rg47749/data/mos2/symm/epsmat_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_dict, f)

In [ ]:
import pickle

with open('/anvil/scratch/x-rg47749/data/mos2/symm/wcoul_mat_dict.pkl', 'wb') as f:
    pickle.dump(wcoul_mat_dict, f)

In [ ]:
ttkw.Eps1.gind_eps2rho[]

In [ ]:
q_vec = ttkw.fbz_q_ind2vec[100]
q_orgid = ttkw.q_ind_fbz2irrbz[100]
print(q_orgid)
#for i in ttkw.Eps1.gind_eps2rho[q_orgid][:700]:   
#    print(epsym_dict[q_vec][i])

In [ ]:
epsym_dict[q_vec]

np.array(sorted([x for x in sorted(epsym_dict[q_vec].keys())]))

In [ ]:
ttkw.Eps1.gind_eps2rho[0, 0]

In [ ]:

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=phi_r.real[:,:,ttkw.fft_nz//2+10], 
                #gridsize=100,
                vmax=0.075,
                vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=phi_r.real[:,:,ttkw.fft_nz//2-10], 
                #gridsize=100,
                vmax=0.075,
                vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
np.sum(fine_data2)

In [ ]:
[x for x in [1,2,3]]

In [ ]:
nmtx = len(epsym_dict[(0.0,0.0,0.0)])
w = np.zeros((nmtx,nmtx))
w_eps2rho = np.array(sorted(epsym_dict[(0.0,0.0,0.0)].keys()))

In [ ]:
np.where(np.round(ttkw.fft_kx_tt, 8)==0.66666667)
